In [1]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.stats import norm
from scipy.spatial import cKDTree

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    Matern,
    WhiteKernel
)

# --------------------------------------------------
# Function 3 - Week 7
# --------------------------------------------------
# The Week 7 .npy files already contain the original
# Function 3 data plus Weeks 1-6 exactly once.
#
# Strategy:
# - fit GP to all accumulated observations
# - inspect ARD lengthscales
# - derive candidate widths from the fitted GP
# - generate local + wider + global candidates
# - compare EI and UCB before choosing the Week 7 query

In [2]:
X = np.load("function3/initial_inputs.npy")
Y = np.load("function3/initial_outputs.npy").reshape(-1)

print("X shape:", X.shape)
print("Y shape:", Y.shape)

assert len(X) == len(Y)
assert X.shape[1] == 3

best_idx = np.argmax(Y)
best_x = X[best_idx]
best_y = Y[best_idx]

print("\nCurrent best observed input:", best_x)
print("Current best observed output:", best_y)

X shape: (21, 3)
Y shape: (21,)

Current best observed input: [0.364352 0.404312 0.451822]
Current best observed output: -0.0149952064996286


In [3]:
kernel = (
    ConstantKernel(1.0, (1e-3, 1e3))
    * Matern(
        length_scale=np.ones(3) * 0.2,
        length_scale_bounds=(0.01, 2.0),
        nu=2.5
    )
    + WhiteKernel(
        noise_level=1e-5,
        noise_level_bounds=(1e-8, 1e-1)
    )
)

gp = GaussianProcessRegressor(
    kernel=kernel,
    normalize_y=True,
    n_restarts_optimizer=20,
    random_state=42
)

gp.fit(X, Y)

print("\nFitted kernel:")
print(gp.kernel_)


Fitted kernel:
1.69**2 * Matern(length_scale=[0.807, 1.32, 0.24], nu=2.5) + WhiteKernel(noise_level=1e-08)


/opt/conda/envs/anaconda-2025.12-py312/lib/python3.12/site-packages/sklearn/gaussian_process/kernels.py:440: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-08. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(


In [4]:
lengthscales = gp.kernel_.k1.k2.length_scale

inverse_ls = 1 / lengthscales
sensitivity = inverse_ls / inverse_ls.sum()

print("\nARD lengthscales:", lengthscales)
print("Normalised inverse-lengthscale sensitivity:", sensitivity)


ARD lengthscales: [0.80733693 1.32496129 0.24024773]
Normalised inverse-lengthscale sensitivity: [0.2012168  0.12260717 0.67617603]


In [5]:
local_scale = np.clip(
    0.25 * lengthscales,
    0.02,
    0.10
)

wide_scale = np.clip(
    0.50 * lengthscales,
    0.05,
    0.20
)

print("\nLocal search widths:", local_scale)
print("Wider search widths:", wide_scale)


Local search widths: [0.1        0.1        0.06006193]
Wider search widths: [0.2        0.2        0.12012387]


In [6]:
rng = np.random.default_rng(42)

local_candidates = best_x + rng.normal(
    0,
    local_scale,
    size=(40000, 3)
)

wide_candidates = best_x + rng.normal(
    0,
    wide_scale,
    size=(20000, 3)
)

global_candidates = rng.uniform(
    0,
    1,
    size=(10000, 3)
)

candidates = np.vstack([
    local_candidates,
    wide_candidates,
    global_candidates
])

candidates = np.clip(candidates, 0, 1)

print("Generated candidates:", len(candidates))

Generated candidates: 70000


In [7]:
tree = cKDTree(X)

distance, _ = tree.query(candidates, k=1)

candidates = candidates[distance > 0.01]

print("Candidates after filtering:", len(candidates))

Candidates after filtering: 69969


In [8]:
mu, sigma = gp.predict(
    candidates,
    return_std=True
)

In [9]:
def expected_improvement(mu, sigma, best_y, xi=0.0):

    improvement = mu - best_y - xi

    Z = np.zeros_like(mu)

    valid = sigma > 1e-12
    Z[valid] = improvement[valid] / sigma[valid]

    EI = np.zeros_like(mu)

    EI[valid] = (
        improvement[valid] * norm.cdf(Z[valid])
        + sigma[valid] * norm.pdf(Z[valid])
    )

    return EI

In [10]:
print("\nEI calibration:\n")

for xi in [0.0, 0.001, 0.005, 0.01, 0.02]:

    EI_test = expected_improvement(
        mu,
        sigma,
        best_y,
        xi
    )

    idx = np.argmax(EI_test)

    print(
        f"xi={xi}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n EI =", round(EI_test[idx], 8),
        "\n"
    )


EI calibration:

xi=0.0 
 candidate = [0.32911152 1.         0.55367666] 
 mean = 0.003421 
 std = 0.057545 
 EI = 0.03333085 

xi=0.001 
 candidate = [0.32911152 1.         0.55367666] 
 mean = 0.003421 
 std = 0.057545 
 EI = 0.03270862 

xi=0.005 
 candidate = [0.32911152 1.         0.55367666] 
 mean = 0.003421 
 std = 0.057545 
 EI = 0.03028626 

xi=0.01 
 candidate = [0.32911152 1.         0.55367666] 
 mean = 0.003421 
 std = 0.057545 
 EI = 0.02741023 

xi=0.02 
 candidate = [0.23082445 1.         0.55235032] 
 mean = 0.000219 
 std = 0.061953 
 EI = 0.02239608 



In [11]:
print("\nUCB calibration:\n")

for beta in [0.1, 0.25, 0.5, 1.0, 1.5]:

    UCB = mu + beta * sigma
    idx = np.argmax(UCB)

    print(
        f"beta={beta}",
        "\n candidate =", candidates[idx],
        "\n mean =", round(mu[idx], 6),
        "\n std =", round(sigma[idx], 6),
        "\n UCB =", round(UCB[idx], 6),
        "\n"
    )


UCB calibration:

beta=0.1 
 candidate = [0.32003254 0.79112217 0.52076634] 
 mean = 0.00848 
 std = 0.036961 
 UCB = 0.012176 

beta=0.25 
 candidate = [0.33712176 0.8733752  0.53617612] 
 mean = 0.007658 
 std = 0.045036 
 UCB = 0.018917 

beta=0.5 
 candidate = [0.32911152 1.         0.55367666] 
 mean = 0.003421 
 std = 0.057545 
 UCB = 0.032193 

beta=1.0 
 candidate = [0.23082445 1.         0.55235032] 
 mean = 0.000219 
 std = 0.061953 
 UCB = 0.062171 

beta=1.5 
 candidate = [0.23619658 0.96873131 0.59139579] 
 mean = -0.003451 
 std = 0.065463 
 UCB = 0.094744 



In [12]:
mean_idx = np.argmax(mu)

print("\nHighest predicted mean:")
print("candidate =", candidates[mean_idx])
print("mean =", mu[mean_idx])
print("std =", sigma[mean_idx])


Highest predicted mean:
candidate = [0.32003254 0.79112217 0.52076634]
mean = 0.008479931697493198
std = 0.036960770813170646


In [13]:
# --------------------------------------------------
# Final Function 3 Week 7 selection
# --------------------------------------------------
#
# The acquisition calibration showed two behaviours:
# higher-exploration EI/UCB settings moved towards x2=1 because of
# increased predictive uncertainty, while the highest GP predicted mean
# occurred at [0.320033, 0.791122, 0.520766].
#
# UCB with beta=0.1 independently selected the same point.
#
# The fitted ARD Matern kernel gave lengthscales
# [0.807, 1.32, 0.24], so the GP currently models x3 as the
# fastest-varying dimension and x2 as relatively smoother.
#
# I therefore use low-exploration UCB (beta=0.1), keeping an uncertainty
# component while prioritising the candidate with the strongest GP mean.

beta = 0.1

UCB = mu + beta * sigma
final_idx = np.argmax(UCB)

week7_candidate = candidates[final_idx]

print("Week 7 Function 3 candidate:")
print(week7_candidate)

print("\nPredicted mean:", mu[final_idx])
print("Predicted std:", sigma[final_idx])
print("UCB:", UCB[final_idx])

portal = "-".join(f"{x:.6f}" for x in week7_candidate)

print("\nPortal format:")
print(portal)

Week 7 Function 3 candidate:
[0.32003254 0.79112217 0.52076634]

Predicted mean: 0.008479931697493198
Predicted std: 0.036960770813170646
UCB: 0.012176008778810263

Portal format:
0.320033-0.791122-0.520766
